In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
movies = pd.read_csv("/content/movies.csv")

In [3]:
ratings = pd.read_csv("/content/ratings.csv")

In [4]:
print("Movies Shape:", movies.shape)

Movies Shape: (27278, 3)


In [5]:
print("Ratings Shape:", ratings.shape)

Ratings Shape: (1048575, 4)


In [6]:
print("\nMovies Sample:")
print(movies.head())


Movies Sample:
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


In [7]:
print("\nRatings Sample:")
print(ratings.head())


Ratings Sample:
   userId  movieId  rating   timestamp
0       1        2     3.5  1112486027
1       1       29     3.5  1112484676
2       1       32     3.5  1112484819
3       1       47     3.5  1112484727
4       1       50     3.5  1112484580


**Data Merging and EDA**

In [9]:
# Merge movies and ratings
merged = pd.merge(ratings, movies, on='movieId', how='left')

print('Merged Shape:', merged.shape)
print('\nSample Merged Data:')
print(merged.head())

Merged Shape: (1048575, 6)

Sample Merged Data:
   userId  movieId  rating   timestamp  \
0       1        2     3.5  1112486027   
1       1       29     3.5  1112484676   
2       1       32     3.5  1112484819   
3       1       47     3.5  1112484727   
4       1       50     3.5  1112484580   

                                               title  \
0                                     Jumanji (1995)   
1  City of Lost Children, The (Cité des enfants p...   
2          Twelve Monkeys (a.k.a. 12 Monkeys) (1995)   
3                        Seven (a.k.a. Se7en) (1995)   
4                         Usual Suspects, The (1995)   

                                   genres  
0              Adventure|Children|Fantasy  
1  Adventure|Drama|Fantasy|Mystery|Sci-Fi  
2                 Mystery|Sci-Fi|Thriller  
3                        Mystery|Thriller  
4                  Crime|Mystery|Thriller  


In [10]:
# Basic Statistics
print("\nAverage Rating:", round(merged['rating'].mean(), 2))
print("Total Users:", merged['userId'].nunique())
print("Total Movies Rated:", merged['movieId'].nunique())


Average Rating: 3.53
Total Users: 7120
Total Movies Rated: 14026


In [11]:
# Popular Movies
popular = merged.groupby('title')['rating'].agg(['mean', 'count']).reset_index()
popular = popular[popular['count'] >= 50].sort_values('mean', ascending = False)

print("\nTop 10 Highest Rated Movies (with min 50 ratings):")
print(popular.head(10)[['title', 'mean', 'count']])


Top 10 Highest Rated Movies (with min 50 ratings):
                                  title      mean  count
11106  Shawshank Redemption, The (1994)  4.469994   3216
5071              Godfather, The (1972)  4.388161   2137
13126        Usual Suspects, The (1995)  4.370482   2490
1098            Band of Brothers (2001)  4.353070    228
2852                      Cosmos (1980)  4.339286     56
2921           Creature Comforts (1989)  4.311475    122
10851           Schindler's List (1993)  4.295612   2598
5072     Godfather: Part II, The (1974)  4.278561   1418
8035                     Matewan (1987)  4.273438     64
7059               Lady Eve, The (1941)  4.269231     78


**User-Movie Matrix and Similarity**

In [12]:
# Create User-Movie Matrix (simple version)
user_movie_matrix = merged.pivot_table(index='userId', columns='title', values='rating').fillna(0)

print("User-Movie Matrix Shape:", user_movie_matrix.shape)
print("\nSample of Matrix (first 5 users, first 5 movies):")
print(user_movie_matrix.iloc[:5, :5])

User-Movie Matrix Shape: (7120, 14021)

Sample of Matrix (first 5 users, first 5 movies):
title   "Great Performances" Cats (1998)  $5 a Day (2008)  '71 (2014)  \
userId                                                                  
1                                    0.0              0.0         0.0   
2                                    0.0              0.0         0.0   
3                                    0.0              0.0         0.0   
4                                    0.0              0.0         0.0   
5                                    0.0              0.0         0.0   

title   'Hellboy': The Seeds of Creation (2004)  \
userId                                            
1                                           0.0   
2                                           0.0   
3                                           0.0   
4                                           0.0   
5                                           0.0   

title   'Neath the Arizona Skies (1934) 

**Calculating Movie Similarity**

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate similarity between movies
movie_similarity = cosine_similarity(user_movie_matrix.T)

print("Movie Similarity Matrix Shape:", movie_similarity.shape)
print("Similarity calculated successfully!")

Movie Similarity Matrix Shape: (14021, 14021)
Similarity calculated successfully!


Recommendation Function

In [14]:
# Recommendation Function

def recommend_similar_movies(movie_name, top_n=10):
    if movie_name not in user_movie_matrix.columns:
        return "Movie not found in dataset"

    # Get index of the movie
    movie_idx = list(user_movie_matrix.columns).index(movie_name)

    # Get similarity scores for this movie
    similarity_scores = movie_similarity[movie_idx]

    # Get top similar movies
    similar_movies = list(enumerate(similarity_scores))
    similar_movies = sorted(similar_movies, key=lambda x: x[1], reverse=True)

    # Return top N (excluding the movie itself)
    recommendations = [user_movie_matrix.columns[i[0]] for i in similar_movies[1:top_n+1]]

    return recommendations

# Test
print("Movies similar to 'Toy Story (1995)':")
print(recommend_similar_movies('Toy Story (1995)', top_n=10))

Movies similar to 'Toy Story (1995)':
['Star Wars: Episode IV - A New Hope (1977)', 'Independence Day (a.k.a. ID4) (1996)', 'Star Wars: Episode VI - Return of the Jedi (1983)', 'Forrest Gump (1994)', 'Mission: Impossible (1996)', 'Toy Story 2 (1999)', 'Back to the Future (1985)', 'Aladdin (1992)', 'Jurassic Park (1993)', 'Willy Wonka & the Chocolate Factory (1971)']


**Hybrid Recommendation**

In [15]:
# Hybrid Recommendation Function
def hybrid_recommend(movie_name, top_n=10):
    if movie_name not in merged['title'].values:
        return "Movie not found"

    # Get genre of the movie
    genre = merged[merged['title'] == movie_name]['genres'].iloc[0]

    # Get index of the movie for similarity
    movie_idx = list(user_movie_matrix.columns).index(movie_name)
    similarity_scores = movie_similarity[movie_idx]

    # Get similar movies based on rating pattern
    similar_movies = list(enumerate(similarity_scores))
    similar_movies = sorted(similar_movies, key=lambda x: x[1], reverse=True)

    # Combine with same genre
    recommendations = []
    for i, score in similar_movies[1:top_n*2]:
        rec_movie = user_movie_matrix.columns[i]
        rec_genre = merged[merged['title'] == rec_movie]['genres'].iloc[0]


        if rec_genre == genre:
            recommendations.append(rec_movie)

        if len(recommendations) >= top_n:
            break


    return recommendations

# Test
print("Hybrid recommendations for 'Toy Story (1995)':")
print(hybrid_recommend('Toy Story (1995)', top_n=10))

Hybrid recommendations for 'Toy Story (1995)':
['Toy Story 2 (1999)']


**Better Hybrid Recommendation**

In [17]:
# Improved Hybrid Recommendation
def hybrid_recommend_improved(movie_name, top_n=10):
    if movie_name not in merged['title'].values:
        return "Movie not found"

    # Get genre of the movie
    genre = merged[merged['title'] == movie_name]['genres'].iloc[0]

    # Get index of the movie
    movie_idx = list(user_movie_matrix.columns).index(movie_name)
    similarity_scores = movie_similarity[movie_idx]

    # Get similar movies
    similar_movies = list(enumerate(similarity_scores))
    similar_movies = sorted(similar_movies, key=lambda x: x[1], reverse=True)

    recommendations = []
    for i, score in similar_movies[1:50]:
        rec_movie = user_movie_matrix.columns[i]
        rec_genre = merged[merged['title'] == rec_movie]['genres'].iloc[0]

        if rec_genre == genre or score > 0.85:
            recommendations.append(rec_movie)

        if len(recommendations) >= top_n:
            break

    return recommendations

# Test
print("Improved Hybrid recommendations for 'Heat (1995)':")
print(hybrid_recommend_improved('Heat (1995)', top_n=10))

Improved Hybrid recommendations for 'Heat (1995)':
['Die Hard (1988)', 'Die Hard: With a Vengeance (1995)', 'Natural Born Killers (1994)', 'Batman (1989)']
